## Step 1: Install dependencies

In [2]:
!pip install sentence-transformers faiss-cpu rank_bm25 pypdf google-genai ddgs -q
print("Done installing")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.4 MB/s eta 0:00:00
Done installing


## Step 2: Download and process the Nepal Constitution PDF

In [3]:
!wget -O nepal_constitution.pdf "https://www.ecoi.net/en/file/local/1125402/1930_1444821984_561625364.pdf"

from pypdf import PdfReader
import re

reader = PdfReader("nepal_constitution.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print("Total characters:", len(full_text))

--2026-09-13 03:46:26--  https://www.ecoi.net/en/file/local/1125402/1930_1444821984_561625364.pdf
Resolving www.ecoi.net (www.ecoi.net)... 188.34.190.110, 2a01:4f8:c17:52d7::1
Connecting to www.ecoi.net (www.ecoi.net)|188.34.190.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4937095 (4.7M) [application/pdf]
Saving to: ‘nepal_constitution.pdf’

nepal_constitution. 100%[===================>]   4.71M  5.65MB/s    in 0.8s    

2026-09-13 03:46:28 (5.65 MB/s) - ‘nepal_constitution.pdf’ saved [4937095/4937095]

Total characters: 340835


## Step 3: Chunk by article and build the hybrid retriever

In [4]:
# structure-aware chunking by article number
article_pattern = re.compile(r'\n\s*(\d{1,3})\.\s+([A-Z][^:]{2,80}):')
matches = list(article_pattern.finditer(full_text))

chunks = []
for i in range(len(matches)):
    start = matches[i].start()
    end = matches[i+1].start() if i + 1 < len(matches) else len(full_text)
    article_text = full_text[start:end].strip()
    if len(article_text) > 20:
        chunks.append(article_text)

print("Number of chunks:", len(chunks))

# build hybrid retriever (embeddings + BM25)
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from rank_bm25 import BM25Okapi

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def retrieve_documents(query: str, k: int = 3) -> str:
    """Retrieve relevant chunks from the Nepal Constitution using hybrid search."""
    try:
        tokenized_query = query.lower().split()
        bm25_scores = bm25.get_scores(tokenized_query)
        bm25_scores_norm = bm25_scores / (bm25_scores.max() + 1e-8)

        query_embedding = embed_model.encode([query])
        distances, indices = index.search(np.array(query_embedding).astype('float32'), k=len(chunks))
        embed_scores = np.zeros(len(chunks))
        max_dist = distances[0].max()
        for rank, idx in enumerate(indices[0]):
            embed_scores[idx] = 1 - (distances[0][rank] / (max_dist + 1e-8))

        combined_scores = 0.5 * bm25_scores_norm + 0.5 * embed_scores
        top_k_idx = np.argsort(combined_scores)[::-1][:k]

        return "\n\n".join([chunks[i] for i in top_k_idx])
    except Exception as e:
        return f"Error: {e}"

# test it
print(retrieve_documents("What does the constitution say about the death penalty?"))

Number of chunks: 285


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

301. Provisions regarding Constitutional Bodies and Officials thereof : (1) The 
Constitutional bodies subsisting at the time of the commencement of this 
Constitution shall be deemed to have been constituted under this 
Constitution, and this Constitution shall not hinder such bodies in dealing with 
matters under consideration according to the existing laws. 
(2) The Chief or officials of the C onstitutional bodies engaged at the time of 
commencement of this constitution shall be deemed to have been 
appointed under this Constitution  and he/she shall remain in his/her 
office subject to the terms and conditions stated at the time of 
appointment. 
(3) If during the commencement of this constitution, there are additional 
number of office-bearers in the Commission for Investigation of Abuse of 
Authority and Public Service Commission than what is specified in this 
constitution, then they shall continue to remain in their office subject to 
the terms and conditions stated at the tim

## Step 4: Diagnose the hybrid search regression

Investigate why combining BM25 + embeddings performs worse than embeddings
alone for this query, by inspecting each score independently.

In [5]:
query = "What does the constitution say about the death penalty?"

# BM25 scores alone
tokenized_query = query.lower().split()
bm25_scores = bm25.get_scores(tokenized_query)
bm25_scores_norm = bm25_scores / (bm25_scores.max() + 1e-8)

top_bm25_idx = np.argsort(bm25_scores)[::-1][:3]
print("=== Top 3 by BM25 alone ===")
for idx in top_bm25_idx:
    print(f"Score: {bm25_scores[idx]:.4f} | {chunks[idx][:100]}")
    print("---")

# embedding scores alone
query_embedding = embed_model.encode([query])
distances, indices = index.search(np.array(query_embedding).astype('float32'), k=3)
print("\n=== Top 3 by embeddings alone ===")
for rank, idx in enumerate(indices[0]):
    print(f"Distance: {distances[0][rank]:.4f} | {chunks[idx][:100]}")
    print("---")

=== Top 3 by BM25 alone ===
Score: 13.1940 | 301. Provisions regarding Constitutional Bodies and Officials thereof : (1) The 
Constitutional bodi
---
Score: 10.7505 | 77. Circumstances under which the Prime Minister and minister ceases to hold 
office: (1) The Prime 
---
Score: 10.6880 | 284. Provision relating to Co nstitutional Council : (1) There shall be a 
Constitutional Council fo
---

=== Top 3 by embeddings alone ===
Distance: 1.1590 | 301. Provisions regarding Constitutional Bodies and Officials thereof : (1) The 
Constitutional bodi
---
Distance: 1.1639 | 16. Right to live with dignity : (1) Each person shall have the rig ht to live with 
dignity. 
(2) N
---
Distance: 1.2609 | 1. Constitution as the fundamental law:  (1) T his constitution is the 
fundamental law of Nepal. Al
---


In [6]:
article_16_idx = [i for i, c in enumerate(chunks) if c.strip().startswith("16.")][0]
print(f"Article 16 BM25 score: {bm25_scores[article_16_idx]:.4f}")
print(f"Article 301 BM25 score: {bm25_scores[301] if len(bm25_scores) > 301 else 'N/A'}")
print(f"Max BM25 score in corpus: {bm25_scores.max():.4f}")

Article 16 BM25 score: 3.8014
Article 301 BM25 score: N/A
Max BM25 score in corpus: 13.1940


## Step 5: Fix — Reciprocal Rank Fusion (RRF) instead of raw score blending

Naive score normalization (divide by max) is fragile: a single high-scoring
but irrelevant BM25 match can skew normalization and bury a genuinely
relevant document that just doesn't share exact vocabulary.

Reciprocal Rank Fusion (RRF) combines *rankings* instead of raw scores —
each document gets 1/(k + rank) from each retrieval method, then these are
summed. This is the standard, more robust approach used in real hybrid
search systems, since it's insensitive to the different score scales and
distributions between BM25 and embedding similarity.

In [7]:
def reciprocal_rank_fusion(query, k=3, rrf_k=60):
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_ranking = np.argsort(bm25_scores)[::-1]

    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k=len(chunks))
    embed_ranking = indices[0]

    rrf_scores = {}
    for rank, idx in enumerate(bm25_ranking):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)
    for rank, idx in enumerate(embed_ranking):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_k + rank)

    top_k_idx = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
    return "\n\n".join([chunks[i] for i in top_k_idx])

# retest the query that failed with naive score blending
print(reciprocal_rank_fusion("What does the constitution say about the death penalty?"))


301. Provisions regarding Constitutional Bodies and Officials thereof : (1) The 
Constitutional bodies subsisting at the time of the commencement of this 
Constitution shall be deemed to have been constituted under this 
Constitution, and this Constitution shall not hinder such bodies in dealing with 
matters under consideration according to the existing laws. 
(2) The Chief or officials of the C onstitutional bodies engaged at the time of 
commencement of this constitution shall be deemed to have been 
appointed under this Constitution  and he/she shall remain in his/her 
office subject to the terms and conditions stated at the time of 
appointment. 
(3) If during the commencement of this constitution, there are additional 
number of office-bearers in the Commission for Investigation of Abuse of 
Authority and Public Service Commission than what is specified in this 
constitution, then they shall continue to remain in their office subject to 
the terms and conditions stated at the tim

In [8]:
tokenized_query = "What does the constitution say about the death penalty?".lower().split()
bm25_scores = bm25.get_scores(tokenized_query)
bm25_ranking = list(np.argsort(bm25_scores)[::-1])

article_16_idx = [i for i, c in enumerate(chunks) if c.strip().startswith("16.")][0]
bm25_rank = bm25_ranking.index(article_16_idx)
print(f"Article 16's rank in BM25: {bm25_rank} out of {len(chunks)}")

query_embedding = embed_model.encode(["What does the constitution say about the death penalty?"])
distances, indices = index.search(np.array(query_embedding).astype('float32'), k=len(chunks))
embed_ranking = list(indices[0])
embed_rank = embed_ranking.index(article_16_idx)
print(f"Article 16's rank in embeddings: {embed_rank} out of {len(chunks)}")

Article 16's rank in BM25: 276 out of 285
Article 16's rank in embeddings: 1 out of 285


In [9]:
article_301_idx = [i for i, c in enumerate(chunks) if c.strip().startswith("301.")][0]
bm25_rank_301 = bm25_ranking.index(article_301_idx)
embed_rank_301 = embed_ranking.index(article_301_idx)
print(f"Article 301's rank in BM25: {bm25_rank_301}")
print(f"Article 301's rank in embeddings: {embed_rank_301}")

rrf_16 = 1/(60+1) + 1/(60+276)
rrf_301 = 1/(60+bm25_rank_301) + 1/(60+embed_rank_301)
print(f"\nArticle 16 RRF score: {rrf_16:.4f}")
print(f"Article 301 RRF score: {rrf_301:.4f}")

Article 301's rank in BM25: 0
Article 301's rank in embeddings: 0

Article 16 RRF score: 0.0194
Article 301 RRF score: 0.0333


## Step 6: Fix — Cross-encoder re-ranking

RRF failed because BM25 and embeddings made the *same* mistake (both
ranked Article 301 above Article 16), so combining rankings couldn't
cancel out a shared error. A cross-encoder solves this differently: instead
of comparing pre-computed query and document embeddings separately, it
takes the query and each candidate document *together* as a single input
and directly scores their relevance — allowing it to actually reason about
whether "capital punishment" and "death penalty" refer to the same concept.

Approach: retrieve a larger candidate set (e.g. top 15 by embeddings),
then re-rank those candidates with a cross-encoder and keep the top 3.

In [10]:
!pip install -q sentence-transformers  # cross-encoder is part of this package

from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_with_reranking(query, initial_k=15, final_k=3):
    # stage 1: cast a wide net with embeddings
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k=initial_k)
    candidates = [chunks[i] for i in indices[0]]

    # stage 2: cross-encoder re-ranks the candidates
    pairs = [[query, doc] for doc in candidates]
    scores = cross_encoder.predict(pairs)

    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return "\n\n".join([doc for doc, score in reranked[:final_k]])

# retest the query that broke both previous approaches
print(retrieve_with_reranking("What does the constitution say about the death penalty?"))

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

16. Right to live with dignity : (1) Each person shall have the rig ht to live with 
dignity. 
(2) No law shall be made for capital punishment.

22. Right against torture:   (1) No person in detention shall be subjected to 
physical or  mental torture, or be treated in a cruel, inhuman or degrading 
manner. 
(2) Any such act pursuant to clause (1) shall be punishable by law and a 
victim of such an act shall have the right to compensation as provided for 
by law. 
Constitution of Nepal 2015, Unofficial English Translation 
 
- 9 -

1. Constitution as the fundamental law:  (1) T his constitution is the 
fundamental law of Nepal. All laws inconsistent with this constitution shall, to 
the extent of such inconsistency, be void. 
(2) It shall be the duty of every person to uphold this constitution.


In [11]:
def retrieve_documents(query: str) -> str:
    """Retrieve relevant chunks from the Nepal Constitution using embedding + cross-encoder re-ranking."""
    try:
        return retrieve_with_reranking(query, initial_k=15, final_k=3)
    except Exception as e:
        return f"Error: {e}"

# quick sanity check
print(retrieve_documents("What does the constitution say about the death penalty?"))

16. Right to live with dignity : (1) Each person shall have the rig ht to live with 
dignity. 
(2) No law shall be made for capital punishment.

22. Right against torture:   (1) No person in detention shall be subjected to 
physical or  mental torture, or be treated in a cruel, inhuman or degrading 
manner. 
(2) Any such act pursuant to clause (1) shall be punishable by law and a 
victim of such an act shall have the right to compensation as provided for 
by law. 
Constitution of Nepal 2015, Unofficial English Translation 
 
- 9 -

1. Constitution as the fundamental law:  (1) T his constitution is the 
fundamental law of Nepal. All laws inconsistent with this constitution shall, to 
the extent of such inconsistency, be void. 
(2) It shall be the duty of every person to uphold this constitution.


In [ ]:
## Step 7: Combine into the full Agentic RAG system

Add retrieve_documents alongside calculator, web_search, and execute_code,
and update the agent's tool description so it knows when to use document
retrieval (questions about the Nepal Constitution specifically) versus web
search (general/current information) versus calculation.

In [13]:
import ast
import operator
import io
import contextlib
import re
from ddgs import DDGS

ALLOWED_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    elif isinstance(node, ast.BinOp):
        return ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    elif isinstance(node, ast.UnaryOp):
        return ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    else:
        raise ValueError(f"Unsupported expression: {node}")

def calculator(expression: str) -> str:
    try:
        tree = ast.parse(expression, mode='eval')
        return str(_safe_eval(tree.body))
    except Exception as e:
        return f"Error: {e}"

def web_search(query: str, max_results: int = 3) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No results found."
        return "\n\n".join([f"{r['title']}: {r['body']}" for r in results])
    except Exception as e:
        return f"Error: {e}"

def execute_code(code: str) -> str:
    output_buffer = io.StringIO()
    try:
        safe_globals = {"__builtins__": {
            "print": print, "range": range, "len": len, "sum": sum,
            "min": min, "max": max, "sorted": sorted, "list": list,
            "dict": dict, "str": str, "int": int, "float": float
        }}
        with contextlib.redirect_stdout(output_buffer):
            exec(code, safe_globals)
        result = output_buffer.getvalue()
        return result if result else "Code executed successfully (no output printed)"
    except Exception as e:
        return f"Error: {e}"

TOOLS_DESCRIPTION = """
You have access to these tools:
1. calculator(expression) - evaluates a math expression, e.g. calculator("47 * 892")
2. web_search(query) - searches the web for current/general information, e.g. web_search("population of Nepal")
3. execute_code(code) - runs Python code and returns printed output, e.g. execute_code("print(sum([1,2,3]))")
4. retrieve_documents(query) - searches the Nepal Constitution specifically for legal/constitutional questions, e.g. retrieve_documents("What does the constitution say about fundamental rights?")

Use retrieve_documents for any question about Nepal's constitution, laws, articles, or legal provisions.
Use web_search for general current information not related to the constitution.
Use calculator for any arithmetic.

Respond in this exact format:
Thought: <your reasoning about what to do next>
Action: <tool_name>(<input>)

Once you have enough information to answer, respond instead with:
Thought: <your reasoning>
Final Answer: <your answer to the user>
"""

def run_tool(action_text):
    match = re.match(r'(\w+)\((.*)\)', action_text.strip(), re.DOTALL)
    if not match:
        return "Error: could not parse action"
    tool_name, tool_input = match.group(1), match.group(2).strip().strip('"').strip("'")

    if tool_name == "calculator":
        return calculator(tool_input)
    elif tool_name == "web_search":
        return web_search(tool_input)
    elif tool_name == "execute_code":
        return execute_code(tool_input)
    elif tool_name == "retrieve_documents":
        return retrieve_documents(tool_input)
    else:
        return f"Error: unknown tool {tool_name}"

def run_agent(question, gemini_client, model="gemini-3.6-flash", max_steps=5):
    history = f"{TOOLS_DESCRIPTION}\n\nQuestion: {question}\n"

    for step in range(max_steps):
        response = gemini_client.models.generate_content(model=model, contents=history)
        text = response.text
        print(f"--- Step {step+1} ---")
        print(text)

        if "Final Answer:" in text:
            return text.split("Final Answer:")[-1].strip()

        action_match = re.search(r'Action:\s*(.+)', text)
        if action_match:
            action_text = action_match.group(1).strip()
            observation = run_tool(action_text)
            print(f"Observation: {observation}\n")
            history += f"{text}\nObservation: {observation}\n"
        else:
            return "Agent did not produce a valid action or final answer."

    return "Max steps reached without a final answer."

print("Agent ready with 4 tools")

Agent ready with 4 tools


In [14]:
from google import genai
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

answer = run_agent("What does the Nepal constitution say about the death penalty?", client)
print("\n=== FINAL ANSWER ===")
print(answer)

--- Step 1 ---
Thought: The user is asking about what the Nepal Constitution says regarding the death penalty. I should search the constitution documents using `retrieve_documents`.

Action: retrieve_documents("death penalty")
Observation: 102. Penalty for Unauthorized Presence or Voting:  If a person sits or votes in a 
meeting of either House of Parliament as a member without taking an oath 
pursuant to Article 88, or knowing that s/he is not qualified for membership in 
the House, s/he shall, on order of the person chairing the House, be liable to 
a fine of five thousand rupees for each day of such presence or voting. The 
fine shall be recovered as government dues.

20. Right to Justice:  (1) No person shall be d etained without being informed of 
the ground for such an arrest. 
(2) The person who is arrested shall have the right to consult a legal 
practitioner of her/his choice and be defended fro m the time of arrest. 
The consultations held with the legal practitioner and the 

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 27.554067328s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},